In [1]:
import time
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt

In [2]:
from google.colab import files
uploaded = files.upload()

Saving a280.tsp to a280.tsp
Saving kroE100.tsp to kroE100.tsp
Saving lin105.tsp to lin105.tsp
Saving tsp225.tsp to tsp225.tsp
Saving st70.tsp to st70.tsp
Saving eil101.tsp to eil101.tsp
Saving ch150.tsp to ch150.tsp
Saving ch130.tsp to ch130.tsp
Saving berlin52.tsp to berlin52.tsp
Saving eil51.tsp to eil51.tsp


In [3]:
#Fonction de lecture des fichiers TSPLIB
def read_tsp(file_name):
    coords = []
    start = False

    with open(file_name, "r") as f:
        for line in f:
            line = line.strip()

            if line == "NODE_COORD_SECTION":
                start = True
                continue

            if line == "EOF":
                break

            if start:
                node, x, y = line.split()
                coords.append((int(node), float(x), float(y)))

    return pd.DataFrame(coords, columns=["node", "x", "y"])

In [4]:
df_eil51 = read_tsp("/content/eil51.tsp")
df_berlin52 = read_tsp("/content/berlin52.tsp")
df_tsp225 = read_tsp("/content/tsp225.tsp")
df_ch130 = read_tsp("/content/ch130.tsp")
df_ch150 = read_tsp("/content/ch150.tsp")
df_st70 = read_tsp("/content/st70.tsp")
df_eil101 = read_tsp("/content/eil101.tsp")
df_kroE100 = read_tsp("/content/kroE100.tsp")
df_lin105 = read_tsp("/content/lin105.tsp")
df_a280 = read_tsp("/content/a280.tsp")

In [30]:
#Fonction pour calculer la matrice de distance
def distance_matrix(df):
    coords = df[["x","y"]].values
    return cdist(coords, coords, metric="euclidean")

In [7]:
dist_eil51 = distance_matrix(df_eil51)
dist_berlin52 = distance_matrix(df_berlin52)
dist_tsp225 = distance_matrix(df_tsp225)
dist_ch130 = distance_matrix(df_ch130)
dist_ch150 = distance_matrix(df_ch150)
dist_st70 = distance_matrix(df_st70)
dist_eil101 = distance_matrix(df_eil101)
dist_kroE100 = distance_matrix(df_kroE100)
dist_lin105 = distance_matrix(df_lin105)
dist_a280 = distance_matrix(df_a280)

In [31]:
#algorithme glouton Nearest neighbor
def nearest_neighbor(distance_matrix, start=0):

    start_time = time.time()

    n = len(distance_matrix)
    visited = [False] * n
    tour = [start]

    visited[start] = True
    current = start
    cost = 0

    for _ in range(n - 1):

        nearest = None
        nearest_dist = float("inf")

        for j in range(n):
            if not visited[j] and distance_matrix[current][j] < nearest_dist:
                nearest = j
                nearest_dist = distance_matrix[current][j]

        tour.append(nearest)
        visited[nearest] = True
        cost += nearest_dist
        current = nearest

    # fermer la tournée
    cost += distance_matrix[current][start]
    tour.append(start)

    end_time = time.time()

    return {
        "tour": tour,
        "cost": cost,
        "time": end_time - start_time
    }

In [9]:
distance_matrices = {
    "eil51": dist_eil51,
    "berlin52": dist_berlin52,
    "st70": dist_st70,
    "kroE100": dist_kroE100,
    "eil101": dist_eil101,
    "lin105": dist_lin105,
    "ch130": dist_ch130,
    "ch150": dist_ch150,
    "tsp225": dist_tsp225,
    "a280": dist_a280
}

In [10]:
results = {}

for name, dist_matrix in distance_matrices.items():

    result = nearest_neighbor(dist_matrix, start=0)

    results[name] = result

    print("Instance:", name)
    print("Tour cost:", result["cost"])
    print("Time:", result["time"])
    print("-"*40)

Instance: eil51
Tour cost: 513.610006884723
Time: 0.0005044937133789062
----------------------------------------
Instance: berlin52
Tour cost: 8980.918279329191
Time: 0.0005450248718261719
----------------------------------------
Instance: st70
Tour cost: 805.5312008437772
Time: 0.0008945465087890625
----------------------------------------
Instance: kroE100
Tour cost: 27587.187458190685
Time: 0.0020329952239990234
----------------------------------------
Instance: eil101
Tour cost: 825.2423227277445
Time: 0.0017735958099365234
----------------------------------------
Instance: lin105
Tour cost: 20362.75741726528
Time: 0.0019512176513671875
----------------------------------------
Instance: ch130
Tour cost: 7575.286291798959
Time: 0.003220081329345703
----------------------------------------
Instance: ch150
Tour cost: 8194.614331512179
Time: 0.006267070770263672
----------------------------------------
Instance: tsp225
Tour cost: 4828.9966828442675
Time: 0.008430004119873047
----------

In [12]:
summary = []

for name, result in results.items():

    summary.append({
        "Instance": name,
        "Cost": result["cost"],
        "Time (s)": result["time"]
    })

df_results = pd.DataFrame(summary)

df_results

,Instance,Cost,Time (s)
0,eil51,513.610007,0.000504
1,berlin52,8980.918279,0.000545
2,st70,805.531201,0.000895
3,kroE100,27587.187458,0.002033
4,eil101,825.242323,0.001774
5,lin105,20362.757417,0.001951
6,ch130,7575.286292,0.003220
7,ch150,8194.614332,0.006267
8,tsp225,4828.996683,0.008430
9,a280,3148.109935,0.012883


In [13]:
greedy_results = results.copy()

In [14]:
greedy_tour = greedy_results[name]["tour"][:-1]

**2 - PSO - warm start (greedy)**

In [15]:
def tour_cost(tour, dist):
    cost = 0
    for i in range(len(tour) - 1):
        cost += dist[tour[i]][tour[i+1]]
    cost += dist[tour[-1]][tour[0]]
    return cost

In [16]:
def random_tour(n):
    tour = list(range(n))
    np.random.shuffle(tour)
    return tour

In [17]:
def two_opt(tour, dist):

    n = len(tour)
    improved = True

    while improved:
        improved = False

        for i in range(1, n-2):
            for j in range(i+1, n):

                a, b = tour[i-1], tour[i]
                c, d = tour[j-1], tour[j % n]

                before = dist[a][b] + dist[c][d]
                after  = dist[a][c] + dist[b][d]

                if after < before:
                    tour[i:j] = reversed(tour[i:j])
                    improved = True

    return tour

In [18]:
def pso_tsp(dist_matrix, greedy_tour, num_particles=20, iterations=200):

    start_time = time.time()

    n = len(dist_matrix)

    particles = []

    # first particle = greedy solution
    particles.append(greedy_tour.copy())

    # remaining particles = slight variations
    for _ in range(num_particles - 1):
        new = greedy_tour.copy()

        i, j = np.random.randint(0, n, 2)
        new[i], new[j] = new[j], new[i]

        particles.append(new)

    pbest = particles[:]
    pbest_cost = [tour_cost(p, dist_matrix) for p in particles]

    gbest = pbest[np.argmin(pbest_cost)]
    gbest_cost = min(pbest_cost)

    for _ in range(iterations):

        for i in range(num_particles):

            new_particle = particles[i][:]

            a, b = np.random.randint(0, n, 2)
            new_particle[a], new_particle[b] = new_particle[b], new_particle[a]

            # optional local search
            if np.random.rand() < 0.3:
                new_particle = two_opt(new_particle, dist_matrix)

            cost = tour_cost(new_particle, dist_matrix)

            particles[i] = new_particle

            if cost < pbest_cost[i]:
                pbest[i] = new_particle
                pbest_cost[i] = cost

        best_idx = np.argmin(pbest_cost)

        if pbest_cost[best_idx] < gbest_cost:
            gbest = pbest[best_idx]
            gbest_cost = pbest_cost[best_idx]

    end_time = time.time()

    return {
        "tour": gbest,
        "cost": gbest_cost,
        "time": end_time - start_time
    }

In [19]:
pso_results = {}

for name, dist_matrix in distance_matrices.items():

    greedy_tour = greedy_results[name]["tour"][:-1]

    result_pso = pso_tsp(dist_matrix, greedy_tour)

    pso_results[name] = result_pso

    print("Instance:", name)
    print("Cost:", result_pso["cost"])
    print("Time:", result_pso["time"])
    print("-"*40)

Instance: eil51
Cost: 436.9059675158406
Time: 7.478546142578125
----------------------------------------
Instance: berlin52
Cost: 7622.020337534719
Time: 9.128687620162964
----------------------------------------
Instance: st70
Cost: 679.1850459676073
Time: 16.618428707122803
----------------------------------------
Instance: kroE100
Cost: 22384.614247924867
Time: 38.27655267715454
----------------------------------------
Instance: eil101
Cost: 663.4239577374658
Time: 37.292450189590454
----------------------------------------
Instance: lin105
Cost: 14751.830646346996
Time: 41.21473479270935
----------------------------------------
Instance: ch130
Cost: 6377.135014329591
Time: 69.40367794036865
----------------------------------------
Instance: ch150
Cost: 6765.443098770766
Time: 90.23144125938416
----------------------------------------
Instance: tsp225
Cost: 4107.119329781988
Time: 215.20253682136536
----------------------------------------
Instance: a280
Cost: 2686.9758904573255
Tim

In [32]:
comparison = []

for name in greedy_results.keys():

    greedy_cost = greedy_results[name]["cost"]
    greedy_time = greedy_results[name]["time"]

    pso_cost = pso_results[name]["cost"]
    pso_time = pso_results[name]["time"]

    gain = 100 * (greedy_cost - pso_cost) / greedy_cost

    comparison.append({
        "Instance": name,
        "Greedy Cost": greedy_cost,
        "PSO (warm start) Cost": pso_cost,
        "Gain (%)": gain,
        "Greedy Time (s)": greedy_time,
        "PSO Time (warm start) (s)": pso_time
    })

In [33]:
df_comparison = pd.DataFrame(comparison)

df_comparison["Gain (%)"] = df_comparison["Gain (%)"].round(2)

df_comparison

,Instance,Greedy Cost,PSO (warm start) Cost,Gain (%),Greedy Time (s),PSO Time (warm start) (s)
0,eil51,513.610007,436.905968,14.93,0.000504,7.478546
1,berlin52,8980.918279,7622.020338,15.13,0.000545,9.128688
2,st70,805.531201,679.185046,15.68,0.000895,16.618429
3,kroE100,27587.187458,22384.614248,18.86,0.002033,38.276553
4,eil101,825.242323,663.423958,19.61,0.001774,37.292450
5,lin105,20362.757417,14751.830646,27.55,0.001951,41.214735
6,ch130,7575.286292,6377.135014,15.82,0.003220,69.403678
7,ch150,8194.614332,6765.443099,17.44,0.006267,90.231441
8,tsp225,4828.996683,4107.119330,14.95,0.008430,215.202537
9,a280,3148.109935,2686.975890,14.65,0.012883,365.494842


**GRASP - Greedy Randomized Adaptive Search Procedure**

In [39]:
def grasp_tsp(dist_matrix, iterations=100, alpha=0.3):

    start_time = time.time()

    n = len(dist_matrix)

    best_tour = None
    best_cost = float("inf")

    for _ in range(iterations):

        # ----- Greedy randomized construction -----
        unvisited = list(range(n))
        current = np.random.choice(unvisited)

        tour = [current]
        unvisited.remove(current)

        while unvisited:

            costs = [(j, dist_matrix[current][j]) for j in unvisited]
            costs.sort(key=lambda x: x[1])

            min_cost = costs[0][1]
            max_cost = costs[-1][1]

            threshold = min_cost + alpha * (max_cost - min_cost)

            rcl = [city for city, c in costs if c <= threshold]

            next_city = np.random.choice(rcl)

            tour.append(next_city)
            unvisited.remove(next_city)

            current = next_city

        # ----- Local search -----
        tour = two_opt(tour, dist_matrix)

        cost = tour_cost(tour, dist_matrix)

        if cost < best_cost:
            best_cost = cost
            best_tour = tour

    end_time = time.time()

    return {
        "tour": best_tour,
        "cost": best_cost,
        "time": end_time - start_time
    }

Approche expérimental, on calcule le tour en utilisant des differentes valeurs pour Alpha, l'Alpha en GRASP contrôle l'équilibre entre comportement avide et exploration aléatoire durant la phase de construction.

In [40]:
alphas = [0.3, 0.4, 0.5]

grasp_results = {}

for alpha in alphas:

    grasp_results[alpha] = {}

    print(f"\n===== ALPHA = {alpha} =====")

    for name, dist_matrix in distance_matrices.items():

        result_grasp = grasp_tsp(dist_matrix, alpha=alpha)

        grasp_results[alpha][name] = result_grasp

        print("Instance:", name)
        print("Cost:", result_grasp["cost"])
        print("Time:", result_grasp["time"])
        print("-"*40)


===== ALPHA = 0.3 =====
Instance: eil51
Cost: 439.3590143043667
Time: 1.8257122039794922
----------------------------------------
Instance: berlin52
Cost: 7756.553317968861
Time: 1.273414134979248
----------------------------------------
Instance: st70
Cost: 709.5415175868103
Time: 1.9812264442443848
----------------------------------------
Instance: kroE100
Cost: 23332.663402689624
Time: 4.46092414855957
----------------------------------------
Instance: eil101
Cost: 683.0137923356012
Time: 6.062082290649414
----------------------------------------
Instance: lin105
Cost: 14841.081617735364
Time: 4.847067594528198
----------------------------------------
Instance: ch130
Cost: 6491.189945942531
Time: 9.35961389541626
----------------------------------------
Instance: ch150
Cost: 6883.385825388109
Time: 12.286656379699707
----------------------------------------
Instance: tsp225
Cost: 4225.986751484684
Time: 28.16719388961792
----------------------------------------
Instance: a280
Cost:

In [41]:
comparison = []

for alpha in alphas:

    for name in greedy_results.keys():

        greedy_cost = greedy_results[name]["cost"]
        greedy_time = greedy_results[name]["time"]

        grasp_cost = grasp_results[alpha][name]["cost"]
        grasp_time = grasp_results[alpha][name]["time"]

        gain = 100 * (greedy_cost - grasp_cost) / greedy_cost

        comparison.append({
            "Instance": name,
            "Alpha": alpha,
            "Greedy Cost": greedy_cost,
            "GRASP Cost": grasp_cost,
            "Gain (%)": gain,
            "Greedy Time (s)": greedy_time,
            "GRASP Time (s)": grasp_time
        })

In [43]:
df_grasp = pd.DataFrame(comparison)

df_grasp["Gain (%)"] = df_grasp["Gain (%)"].round(2)

df_grasp = df_grasp.sort_values(["Instance", "Alpha"])
df_grasp

,Instance,Alpha,Greedy Cost,GRASP Cost,Gain (%),Greedy Time (s),GRASP Time (s)
9,a280,0.3,3148.109935,2822.629314,10.34,0.012883,49.303891
19,a280,0.4,3148.109935,2824.082627,10.29,0.012883,45.681729
29,a280,0.5,3148.109935,2869.129568,8.86,0.012883,46.340799
1,berlin52,0.3,8980.918279,7756.553318,13.63,0.000545,1.273414
11,berlin52,0.4,8980.918279,7669.619657,14.60,0.000545,1.117820
21,berlin52,0.5,8980.918279,7872.501021,12.34,0.000545,1.075547
6,ch130,0.3,7575.286292,6491.189946,14.31,0.003220,9.359614
16,ch130,0.4,7575.286292,6540.841239,13.66,0.003220,9.571720
26,ch130,0.5,7575.286292,6517.579539,13.96,0.003220,9.214397
7,ch150,0.3,8194.614332,6883.385825,16.00,0.006267,12.286656


Meilleur Gain par Alpha pour chaque instance :

In [44]:
df_grasp.loc[df_grasp.groupby("Instance")["GRASP Cost"].idxmin()]

,Instance,Alpha,Greedy Cost,GRASP Cost,Gain (%),Greedy Time (s),GRASP Time (s)
9,a280,0.3,3148.109935,2822.629314,10.34,0.012883,49.303891
11,berlin52,0.4,8980.918279,7669.619657,14.60,0.000545,1.117820
6,ch130,0.3,7575.286292,6491.189946,14.31,0.003220,9.359614
7,ch150,0.3,8194.614332,6883.385825,16.00,0.006267,12.286656
4,eil101,0.3,825.242323,683.013792,17.23,0.001774,6.062082
0,eil51,0.3,513.610007,439.359014,14.46,0.000504,1.825712
13,kroE100,0.4,27587.187458,23182.263240,15.97,0.002033,4.936388
5,lin105,0.3,20362.757417,14841.081618,27.12,0.001951,4.847068
12,st70,0.4,805.531201,691.999034,14.09,0.000895,1.993117
28,tsp225,0.5,4828.996683,4188.076202,13.27,0.008430,28.693459


In [47]:
best_grasp = df_grasp.loc[df_grasp.groupby("Instance")["GRASP Cost"].idxmin()]

In [48]:
df_greedy = pd.DataFrame([
    {
        "Instance": name,
        "Greedy Cost": greedy_results[name]["cost"],
        "Greedy Time": greedy_results[name]["time"]
    }
    for name in greedy_results
])

In [49]:
df_pso = pd.DataFrame([
    {
        "Instance": name,
        "PSO Cost": pso_results[name]["cost"],
        "PSO Time": pso_results[name]["time"]
    }
    for name in pso_results
])

In [50]:
best_grasp = best_grasp[["Instance", "Alpha", "GRASP Cost", "GRASP Time (s)"]]
best_grasp = best_grasp.rename(columns={"GRASP Time (s)": "GRASP Time"})

**Tableau Comparatif final**


In [52]:
df_final = df_greedy.merge(df_pso, on="Instance").merge(best_grasp, on="Instance")

df_final = df_final.sort_values("Instance")
df_final.sort_index()

,Instance,Greedy Cost,Greedy Time,PSO Cost,PSO Time,Alpha,GRASP Cost,GRASP Time
0,eil51,513.610007,0.000504,436.905968,7.478546,0.3,439.359014,1.825712
1,berlin52,8980.918279,0.000545,7622.020338,9.128688,0.4,7669.619657,1.117820
2,st70,805.531201,0.000895,679.185046,16.618429,0.4,691.999034,1.993117
3,kroE100,27587.187458,0.002033,22384.614248,38.276553,0.4,23182.263240,4.936388
4,eil101,825.242323,0.001774,663.423958,37.292450,0.3,683.013792,6.062082
5,lin105,20362.757417,0.001951,14751.830646,41.214735,0.3,14841.081618,4.847068
6,ch130,7575.286292,0.003220,6377.135014,69.403678,0.3,6491.189946,9.359614
7,ch150,8194.614332,0.006267,6765.443099,90.231441,0.3,6883.385825,12.286656
8,tsp225,4828.996683,0.008430,4107.119330,215.202537,0.5,4188.076202,28.693459
9,a280,3148.109935,0.012883,2686.975890,365.494842,0.3,2822.629314,49.303891
